In [ ]:
"""
Comparaison Q-learning vs Dyna-Q (Softmax) sur MazeMDP (ratio=0), avec:
- tuning Optuna (par taille et par algo)
- logging performance (return) vs env_steps et vs temps
- métriques: AUC_steps, AUC_time, samples-to-threshold, time-to-threshold
- Welch t-test
- figures Matplotlib

Prérequis (pip):
  gymnasium (ou gym) + environnement "MazeMDP-v0" déjà disponible
  numpy optuna scipy matplotlib

Notes:
- On mesure la performance par RETURN moyen sur des épisodes d'évaluation (policy greedy).
- "env_steps" = nb de transitions collectées dans l'environnement (sample efficiency).
- "time" = temps wall-clock cumulé (perf_counter).
"""
from __future__ import annotations
from bbrl_gymnasium.envs.maze_mdp import MazeMDPEnv

from mazemdp.toolbox import egreedy, egreedy_loc, sample_categorical, softmax
from mazemdp import random_policy

import pandas as pd
import time
import math
import random
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

import optuna
optuna.logging.set_verbosity(optuna.logging.INFO)

# -----------------------------
# 0) Reproductibilité
# -----------------------------
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


# -----------------------------
# 1) Environnement (3 MDP ratio=0)
# -----------------------------
def make_mdp(width: int, height: int, render_mode=None):
    # Compatible gymnasium/gym (selon ton setup)
    import gymnasium as gym

    mdp = gym.make(
        "MazeMDP-v0",
        kwargs={
            "width": width,
            "height": height,
            "ratio": 0.0,       # aucun mur interne
            "hit": 0.0,
            "start_states": [0],  # départ fixe
        },
        render_mode=render_mode,
    )
    env = mdp.unwrapped
    return mdp, env


# -----------------------------
# 2) Outils RL: politiques
# -----------------------------
def egreedy_action(Q: np.ndarray, s: int, epsilon: float, rng: np.random.Generator) -> int:
    nA = Q.shape[1]
    if rng.random() < epsilon:
        return int(rng.integers(0, nA))
    return int(np.argmax(Q[s]))


def greedy_action(Q: np.ndarray, s: int) -> int:
    return int(np.argmax(Q[s]))


def softmax_probs(Q: np.ndarray, s: int, beta: float) -> np.ndarray:
    # beta grand => plus greedy
    x = beta * Q[s]
    x = x - np.max(x)  # stabilité numérique
    ex = np.exp(x)
    p = ex / np.sum(ex)
    return p


def sample_categorical(p: np.ndarray, rng: np.random.Generator) -> int:
    return int(rng.choice(len(p), p=p))


# -----------------------------
# 3) Évaluation (return)
# -----------------------------
def eval_return(mdp, env, Q: np.ndarray, n_eval_episodes: int, seed: int) -> float:
    """
    Évaluation greedy, mais RETURN DISCOUNTED:
    G = sum_t gamma^t * r_t
    => rend les courbes beaucoup plus informatives en reward sparse.
    """
    gamma = float(env.gamma)
    rng = np.random.default_rng(seed)
    returns = []

    for _ in range(n_eval_episodes):
        try:
            s, _ = mdp.reset(seed=int(rng.integers(0, 2**31 - 1)))
        except TypeError:
            s = mdp.reset()
            if isinstance(s, tuple):
                s = s[0]

        done = False
        trunc = False
        G = 0.0
        t = 0

        while not (done or trunc):
            a = int(np.argmax(Q[int(s)]))
            step_out = mdp.step(a)

            if len(step_out) == 5:
                s2, r, done, trunc, _ = step_out
            else:
                s2, r, done, _ = step_out
                trunc = False

            # discounted return
            G += (gamma ** t) * float(r)
            t += 1
            s = s2

        returns.append(G)

    return float(np.mean(returns))

# -----------------------------
# 4) Q-learning (training + logging)
# -----------------------------
@dataclass
class TrainLog:
    steps: np.ndarray         # shape [T]
    times: np.ndarray         # shape [T]
    returns: np.ndarray       # shape [T]
    Q: np.ndarray             # learned Q


def train_q_learning(
    mdp,
    env,
    alpha: float,
    epsilon: float,
    n_env_steps: int,
    eval_every: int,
    n_eval_episodes: int,
    seed: int,
) -> TrainLog:
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    nS = int(env.nb_states)
    nA = int(env.action_space.n)
    gamma = float(env.gamma)

    Q = np.zeros((nS, nA), dtype=float)

    # init
    try:
        s, _ = mdp.reset(seed=seed)
    except TypeError:
        s = mdp.reset()
        if isinstance(s, tuple):
            s = s[0]
    s = int(s)

    t0 = time.perf_counter()

    steps_list, times_list, rets_list = [], [], []

    env_steps = 0
    while env_steps < n_env_steps:
        a = egreedy_action(Q, s, epsilon, rng)

        step_out = mdp.step(int(a))
        if len(step_out) == 5:
            s2, r, terminated, truncated, _ = step_out
            done = bool(terminated)
            trunc = bool(truncated)
        else:
            s2, r, done, info = step_out
            trunc = False

        s2 = int(s2)
        r = float(r)

        # Q-learning update (off-policy)
        target = r if (done or trunc) else (r + gamma * float(np.max(Q[s2])))
        Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])

        env_steps += 1
        if (env_steps % eval_every) == 0 or env_steps == n_env_steps:
            wall_t = time.perf_counter() - t0
            R = eval_return(mdp, env, Q, n_eval_episodes, seed=seed + 10_000 + env_steps)
            steps_list.append(env_steps)
            times_list.append(wall_t)
            rets_list.append(R)

        if done or trunc:
            try:
                s, _ = mdp.reset(seed=int(rng.integers(0, 2**31 - 1)))
            except TypeError:
                s = mdp.reset()
                if isinstance(s, tuple):
                    s = s[0]
            s = int(s)
        else:
            s = s2

    return TrainLog(
        steps=np.asarray(steps_list, dtype=int),
        times=np.asarray(times_list, dtype=float),
        returns=np.asarray(rets_list, dtype=float),
        Q=Q,
    )


# -----------------------------
# 5) Dyna-Q + Softmax (modèle + planning)
# -----------------------------
class DeterministicTransitionModel:
    def __init__(self, nS: int, nA: int):
        self.nS, self.nA = nS, nA
        self.count = np.zeros((nS, nA, nS), dtype=np.int64)
        self.sa_seen: List[Tuple[int, int]] = []

    def add(self, s: int, a: int, s2: int) -> None:
        if self.count[s, a].sum() == 0:
            self.sa_seen.append((s, a))
        self.count[s, a, s2] += 1

    def predict(self, s: int, a: int) -> int:
        c = self.count[s, a]
        if c.sum() == 0:
            return s
        return int(np.argmax(c))

    def sample_sa(self, rng: np.random.Generator) -> Tuple[int, int]:
        # sample uniformly among seen state-actions
        if not self.sa_seen:
            # fallback: arbitrary
            return 0, 0
        idx = int(rng.integers(0, len(self.sa_seen)))
        return self.sa_seen[idx]


class RewardModel:
    def __init__(self, nS: int, nA: int):
        self.R = np.zeros((nS, nA), dtype=float)

    def add(self, s: int, a: int, r: float) -> None:
        self.R[s, a] = r

    def predict(self, s: int, a: int) -> float:
        return float(self.R[s, a])


class TerminationModel:
    def __init__(self, nS: int, nA: int):
        self.T = np.zeros((nS, nA), dtype=np.int8)

    def add(self, s: int, a: int, done_or_trunc: bool) -> None:
        self.T[s, a] = 1 if done_or_trunc else 0

    def predict(self, s: int, a: int) -> bool:
        return bool(self.T[s, a] == 1)


@dataclass
class FullModel:
    trans: DeterministicTransitionModel
    rew: RewardModel
    term: TerminationModel


def train_dynaq_softmax(
    mdp,
    env,
    alpha: float,
    beta: float,
    nb_updates: int,
    n_env_steps: int,
    eval_every: int,
    n_eval_episodes: int,
    seed: int,
) -> TrainLog:
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    nS = int(env.nb_states)
    nA = int(env.action_space.n)
    gamma = float(env.gamma)

    Q = np.zeros((nS, nA), dtype=float)

    model = FullModel(
        trans=DeterministicTransitionModel(nS, nA),
        rew=RewardModel(nS, nA),
        term=TerminationModel(nS, nA),
    )

    # init
    try:
        s, _ = mdp.reset(seed=seed)
    except TypeError:
        s = mdp.reset()
        if isinstance(s, tuple):
            s = s[0]
    s = int(s)

    t0 = time.perf_counter()
    steps_list, times_list, rets_list = [], [], []

    env_steps = 0
    while env_steps < n_env_steps:
        # Softmax exploration for real interaction
        p = softmax_probs(Q, s, beta=beta)
        a = sample_categorical(p, rng)

        step_out = mdp.step(int(a))
        if len(step_out) == 5:
            s2, r, terminated, truncated, _ = step_out
            done = bool(terminated)
            trunc = bool(truncated)
        else:
            s2, r, done, info = step_out
            trunc = False

        s2 = int(s2)
        r = float(r)
        done_or_trunc = bool(done or trunc)

        # 1) Learn model
        model.trans.add(s, a, s2)
        model.rew.add(s, a, r)
        model.term.add(s, a, done_or_trunc)

        # 2) Direct RL update (Q-learning)
        target = r if done_or_trunc else (r + gamma * float(np.max(Q[s2])))
        Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])

        # 3) Planning updates
        for _ in range(nb_updates):
            sp, ap = model.trans.sample_sa(rng)
            rp = model.rew.predict(sp, ap)
            s2p = model.trans.predict(sp, ap)
            donep = model.term.predict(sp, ap)

            targ_p = rp if donep else (rp + gamma * float(np.max(Q[s2p])))
            Q[sp, ap] = Q[sp, ap] + alpha * (targ_p - Q[sp, ap])

        env_steps += 1

        if (env_steps % eval_every) == 0 or env_steps == n_env_steps:
            wall_t = time.perf_counter() - t0
            R = eval_return(mdp, env, Q, n_eval_episodes, seed=seed + 20_000 + env_steps)
            steps_list.append(env_steps)
            times_list.append(wall_t)
            rets_list.append(R)

        if done_or_trunc:
            try:
                s, _ = mdp.reset(seed=int(rng.integers(0, 2**31 - 1)))
            except TypeError:
                s = mdp.reset()
                if isinstance(s, tuple):
                    s = s[0]
            s = int(s)
        else:
            s = s2

    return TrainLog(
        steps=np.asarray(steps_list, dtype=int),
        times=np.asarray(times_list, dtype=float),
        returns=np.asarray(rets_list, dtype=float),
        Q=Q,
    )


# -----------------------------
# 6) Métriques: AUC + time/samples-to-threshold
# -----------------------------
def auc_trapz(x: np.ndarray, y: np.ndarray) -> float:
    # assume x is increasing
    return float(np.trapezoid(y, x))


def first_crossing_x(x: np.ndarray, y: np.ndarray, threshold: float) -> Optional[float]:
    """
    Renvoie la plus petite abscisse x telle que y >= threshold.
    Interpolation linéaire entre deux points si besoin.
    Si jamais atteint -> None.
    """
    for i in range(len(x)):
        if y[i] >= threshold:
            if i == 0:
                return float(x[i])
            # interpolate between i-1 and i
            x0, y0 = float(x[i - 1]), float(y[i - 1])
            x1, y1 = float(x[i]), float(y[i])
            if y1 == y0:
                return float(x1)
            t = (threshold - y0) / (y1 - y0)
            t = min(max(t, 0.0), 1.0)
            return float(x0 + t * (x1 - x0))
    return None


@dataclass
class SummaryMetrics:
    auc_steps: float
    auc_time: float
    samples_to_thr: Optional[float]
    time_to_thr: Optional[float]
    final_return: float
    final_time: float


def summarize_run(log: TrainLog, threshold: float) -> SummaryMetrics:
    auc_s = auc_trapz(log.steps, log.returns)
    auc_t = auc_trapz(log.times, log.returns)
    s_thr = first_crossing_x(log.steps, log.returns, threshold)
    t_thr = first_crossing_x(log.times, log.returns, threshold)
    return SummaryMetrics(
        auc_steps=auc_s,
        auc_time=auc_t,
        samples_to_thr=s_thr,
        time_to_thr=t_thr,
        final_return=float(log.returns[-1]),
        final_time=float(log.times[-1]),
    )


# -----------------------------
# 7) Welch t-test (entre algos)
# -----------------------------
def welch_test(a: np.ndarray, b: np.ndarray, alpha: float = 0.05) -> Tuple[float, bool]:
    # ignore nan (cas: threshold jamais atteint)
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]
    stat, p = ttest_ind(a, b, equal_var=False)
    return float(p), bool(p < alpha)


# -----------------------------
# 8) Optuna tuning
# -----------------------------
@dataclass
class BestParams:
    ql: Dict[str, float]
    dyn: Dict[str, float]


def tune_for_mdp(
    mdp,
    env,
    n_trials: int,
    objective_seeds: List[int],
    n_env_steps_train: int,
    eval_every: int,
    n_eval_episodes: int,
) -> BestParams:
    """
    Tune Q-learning puis Dyna-Q(softmax) séparément.
    Objectif: moyenne des returns d'évaluation finale (sur plusieurs seeds) -> stable.
    """
    eval_every_tune = n_env_steps_train   # 1 seule évaluation, à la fin
    n_eval_episodes_tune = 3 
    def obj_ql(trial: optuna.Trial) -> float:
        alpha = trial.suggest_float("alpha", 1e-3, 1.0, log=True)
        epsilon = trial.suggest_float("epsilon", 0.02, 0.4, log=True)

        scores = []
        for sd in objective_seeds:
            log = train_q_learning(
                mdp=mdp, env=env,
                alpha=alpha, epsilon=epsilon,
                n_env_steps=n_env_steps_train,
                eval_every=eval_every_tune,
                n_eval_episodes=n_eval_episodes_tune,
                seed=sd,
            )
            scores.append(float(log.returns[-1]))
        return float(np.mean(scores))

    def obj_dyn(trial: optuna.Trial) -> float:
        alpha = trial.suggest_float("alpha", 1e-3, 1.0, log=True)
        beta = trial.suggest_float("beta", 0.5, 20.0)
        nb_updates = trial.suggest_int("nb_updates", 0, 10)

        scores = []
        for sd in objective_seeds:
            log = train_dynaq_softmax(
                mdp=mdp, env=env,
                alpha=alpha, beta=beta, nb_updates=nb_updates,
                n_env_steps=n_env_steps_train,
                eval_every=eval_every_tune,
                n_eval_episodes=n_eval_episodes_tune,
                seed=sd,
            )
            scores.append(float(log.returns[-1]))
        return float(np.mean(scores))
    def stop_if_optimal(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
        # 1.0 = score max (retour terminal)
        if study.best_value is not None and study.best_value >= 0.999:
            study.stop()
    study_ql = optuna.create_study(direction="maximize")
    study_ql.optimize(obj_ql, n_trials=n_trials, callbacks=[stop_if_optimal], timeout=120)

    study_dyn = optuna.create_study(direction="maximize")
    study_dyn.optimize(obj_dyn, n_trials=n_trials, callbacks=[stop_if_optimal], timeout=120)

    return BestParams(
        ql=study_ql.best_params,
        dyn=study_dyn.best_params,
    )


# -----------------------------
# 9) Expérience complète (multi-seeds) + figures
# -----------------------------
def aggregate_logs(logs: List[TrainLog]) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Suppose que tous les logs ont les mêmes steps (même eval_every + n_env_steps).
    Retourne steps, times_mean, returns_mean, returns_low, returns_high (80% CI via percentiles).
    """
    steps = logs[0].steps
    times = np.stack([lg.times for lg in logs], axis=0)       # [R, T]
    rets = np.stack([lg.returns for lg in logs], axis=0)      # [R, T]

    times_mean = np.mean(times, axis=0)
    returns_mean = np.mean(rets, axis=0)
    low = np.percentile(rets, 10, axis=0)
    high = np.percentile(rets, 90, axis=0)
    return steps, times_mean, returns_mean, low, high


def plot_perf_vs_steps(
    title: str,
    ql_agg,
    dyn_agg,
    outpath: str,
    fig_num: int,
    n_runs: int,
    ci_desc: str,
    p_final: float,
    p_auc_steps: float,
    threshold: float,
):
    steps_q, _, mean_q, low_q, high_q = ql_agg
    steps_d, _, mean_d, low_d, high_d = dyn_agg

    fig = plt.figure()
    ax = plt.gca()

    ax.plot(steps_q, mean_q, label="Q-learning")
    ax.fill_between(steps_q, low_q, high_q, alpha=0.25)

    ax.plot(steps_d, mean_d, label="Dyna-Q Softmax")
    ax.fill_between(steps_d, low_d, high_d, alpha=0.25)

    # (optionnel mais très utile) ligne threshold + optimum
    ax.axhline(threshold, linestyle=":", alpha=0.6)
    ax.axhline(1.0, linestyle="--", alpha=0.6)

    ax.set_xlabel("Env steps (samples)")
    ax.set_ylabel("Return (evaluation)")

    ax.set_title(f"Figure {fig_num} — {title} — performance vs samples")
    ax.legend()
    ax.grid(True, alpha=0.3)

    caption = (
        f"Return moyen d’évaluation (policy greedy) en fonction du nombre de transitions collectées. "
        f"Bande d’incertitude: {ci_desc} sur N={n_runs} runs. "
        f"Seuil indiqué: return={threshold:.2f}. "
        f"Tests de significativité (Welch): p(final return)={p_final:.3g}, p(AUC_steps)={p_auc_steps:.3g}."
    )
    add_caption(fig, caption)

    fig.tight_layout()
    fig.savefig(outpath, dpi=160, bbox_inches="tight")
    plt.close(fig)


def plot_perf_vs_time(
    title: str,
    ql_agg,
    dyn_agg,
    outpath: str,
    fig_num: int,
    n_runs: int,
    ci_desc: str,
    p_auc_time: float,
    p_time_thr: float,
    threshold: float,
):
    _, time_q, mean_q, low_q, high_q = ql_agg
    _, time_d, mean_d, low_d, high_d = dyn_agg

    fig = plt.figure()
    ax = plt.gca()

    ax.plot(time_q, mean_q, label="Q-learning")
    ax.fill_between(time_q, low_q, high_q, alpha=0.25)

    ax.plot(time_d, mean_d, label="Dyna-Q Softmax")
    ax.fill_between(time_d, low_d, high_d, alpha=0.25)

    ax.axhline(threshold, linestyle=":", alpha=0.6)
    ax.axhline(1.0, linestyle="--", alpha=0.6)

    ax.set_xlabel("Wall time (s)")
    ax.set_ylabel("Return (evaluation)")

    ax.set_title(f"Figure {fig_num} — {title} — performance vs time")
    ax.legend()
    ax.grid(True, alpha=0.3)

    caption = (
        f"Return moyen d’évaluation (policy greedy) en fonction du temps de calcul (wall-clock). "
        f"Bande d’incertitude: {ci_desc} sur N={n_runs} runs. "
        f"Seuil indiqué: return={threshold:.2f}. "
        f"Tests de significativité (Welch): p(AUC_time)={p_auc_time:.3g}, p(time-to-threshold)={p_time_thr:.3g}."
    )
    add_caption(fig, caption)

    fig.tight_layout()
    fig.savefig(outpath, dpi=160, bbox_inches="tight")
    plt.close(fig)

def run_benchmark_for_size(
    width: int,
    height: int,
    n_trials: int = 40,
    objective_seeds: List[int] = [0, 1, 2],
    n_env_steps_train: int = 20_000,
    eval_every: int = 500,
    n_eval_episodes: int = 30,
    n_runs: int = 30,
    threshold: float = 0.0,
    alpha_test: float = 0.05,
    out_prefix: str = "results",
) -> None:
    mdp, env = make_mdp(width, height, render_mode=None)
    ci_desc = "intervalle [p10, p90] (percentiles 10%–90%)"
    # 1) Tune
    best = tune_for_mdp(
        mdp=mdp, env=env,
        n_trials=n_trials,
        objective_seeds=objective_seeds,
        n_env_steps_train=n_env_steps_train,
        eval_every=eval_every,
        n_eval_episodes=n_eval_episodes,
    )

    print(f"\n=== Maze {width}x{height} (ratio=0) ===")
    print("Best Q-learning params:", best.ql)
    print("Best Dyna-Q params:", best.dyn)

    # 2) Multi-run logging
    ql_logs, dyn_logs = [], []
    for run in range(n_runs):
        sd = 10_000 + run

        ql_logs.append(
            train_q_learning(
                mdp=mdp, env=env,
                alpha=float(best.ql["alpha"]),
                epsilon=float(best.ql["epsilon"]),
                n_env_steps=n_env_steps_train,
                eval_every=eval_every,
                n_eval_episodes=n_eval_episodes,
                seed=sd,
            )
        )

        dyn_logs.append(
            train_dynaq_softmax(
                mdp=mdp, env=env,
                alpha=float(best.dyn["alpha"]),
                beta=float(best.dyn["beta"]),
                nb_updates=int(best.dyn["nb_updates"]),
                n_env_steps=n_env_steps_train,
                eval_every=eval_every,
                n_eval_episodes=n_eval_episodes,
                seed=sd,
            )
        )
    
    # 3) Aggregate + plots
    ql_agg = aggregate_logs(ql_logs)
    dyn_agg = aggregate_logs(dyn_logs)
    

    # 4) Metrics per run
    ql_metrics = [summarize_run(lg, threshold=threshold) for lg in ql_logs]
    dyn_metrics = [summarize_run(lg, threshold=threshold) for lg in dyn_logs]

    def arr(getter):
        return np.asarray([getter(m) for m in ql_metrics], dtype=float), np.asarray([getter(m) for m in dyn_metrics], dtype=float)

    auc_steps_q, auc_steps_d = arr(lambda m: m.auc_steps)
    auc_time_q,  auc_time_d  = arr(lambda m: m.auc_time)
    final_q,     final_d     = arr(lambda m: m.final_return)

    # threshold metrics may be None => nan
    sthr_q = np.asarray([m.samples_to_thr if m.samples_to_thr is not None else np.nan for m in ql_metrics], dtype=float)
    sthr_d = np.asarray([m.samples_to_thr if m.samples_to_thr is not None else np.nan for m in dyn_metrics], dtype=float)
    tthr_q = np.asarray([m.time_to_thr if m.time_to_thr is not None else np.nan for m in ql_metrics], dtype=float)
    tthr_d = np.asarray([m.time_to_thr if m.time_to_thr is not None else np.nan for m in dyn_metrics], dtype=float)


    p_final, _ = welch_test(final_q, final_d, alpha=alpha_test)
    p_auc_steps, _ = welch_test(auc_steps_q, auc_steps_d, alpha=alpha_test)
    p_auc_time, _  = welch_test(auc_time_q, auc_time_d, alpha=alpha_test)
    p_time_thr, _  = welch_test(tthr_q, tthr_d, alpha=alpha_test)
    
    ci_desc = "intervalle [p10, p90] (percentiles 10%–90%)"
    base = {3: 0, 5: 2, 8: 4}.get(width, 0)  # 3x3->0, 5x5->2, 8x8->4
    fig_steps = base + 1
    fig_time  = base + 2
    plot_perf_vs_steps(
        title=f"Maze {width}x{height}",
        ql_agg=ql_agg,
        dyn_agg=dyn_agg,
        outpath=f"{out_prefix}_maze_{width}x{height}_perf_vs_steps.png",
        fig_num=fig_steps,                 # incrémente selon ton doc
        n_runs=n_runs,
        ci_desc=ci_desc,
        p_final=p_final,
        p_auc_steps=p_auc_steps,
        threshold=threshold,
    )
    
    plot_perf_vs_time(
        title=f"Maze {width}x{height}",
        ql_agg=ql_agg,
        dyn_agg=dyn_agg,
        outpath=f"{out_prefix}_maze_{width}x{height}_perf_vs_time.png",
        fig_num=fig_time,                 # incrémente
        n_runs=n_runs,
        ci_desc=ci_desc,
        p_auc_time=p_auc_time,
        p_time_thr=p_time_thr,
        threshold=threshold,
    )
    
    
    # 5) Welch tests
    def report(metric_name: str, xq: np.ndarray, xd: np.ndarray, higher_is_better: bool = True):
        p, sig = welch_test(xq, xd, alpha=alpha_test)
        mean_q = float(np.nanmean(xq))
        mean_d = float(np.nanmean(xd))

        if higher_is_better:
            better = "Dyna-Q" if mean_d > mean_q else "Q-learning"
        else:
            better = "Dyna-Q" if mean_d < mean_q else "Q-learning"

        print(f"\n[{metric_name}]")
        print(f"  mean(Q-learning) = {mean_q:.6g}")
        print(f"  mean(Dyna-Q)     = {mean_d:.6g}")
        print(f"  Welch p-value    = {p:.6g}  -> significant={sig} (alpha={alpha_test})")
        print(f"  winner (mean)    = {better}")

    # Performance (higher better)
    report("Final return (higher better)", final_q, final_d, higher_is_better=True)
    report("AUC vs steps (higher better)", auc_steps_q, auc_steps_d, higher_is_better=True)
    report("AUC vs time (higher better)", auc_time_q, auc_time_d, higher_is_better=True)

    # Efficiency (lower better) for time-to-threshold and samples-to-threshold
    report(f"Samples-to-threshold={threshold} (lower better)", sthr_q, sthr_d, higher_is_better=False)
    report(f"Time-to-threshold={threshold} (lower better)", tthr_q, tthr_d, higher_is_better=False)
    
    summary = {
        "maze": f"{width}x{height}",
        "mean_final_return_ql": float(np.nanmean(final_q)),
        "mean_final_return_dyn": float(np.nanmean(final_d)),
        "p_final_return": float(p_final),
    
        "mean_auc_steps_ql": float(np.nanmean(auc_steps_q)),
        "mean_auc_steps_dyn": float(np.nanmean(auc_steps_d)),
        "p_auc_steps": float(p_auc_steps),
    
        "mean_auc_time_ql": float(np.nanmean(auc_time_q)),
        "mean_auc_time_dyn": float(np.nanmean(auc_time_d)),
        "p_auc_time": float(p_auc_time),
    
        "mean_samples_to_thr_ql": float(np.nanmean(sthr_q)),
        "mean_samples_to_thr_dyn": float(np.nanmean(sthr_d)),
    
        "mean_time_to_thr_ql": float(np.nanmean(tthr_q)),
        "mean_time_to_thr_dyn": float(np.nanmean(tthr_d)),
        "p_time_to_thr": float(p_time_thr),
    }
    df = pd.DataFrame([summary])
    df.to_csv(f"{out_prefix}_maze_{width}x{height}_summary.csv", index=False)
    
    print("\nSaved figures:")
    print(f"  {out_prefix}_maze_{width}x{height}_perf_vs_steps.png")
    print(f"  {out_prefix}_maze_{width}x{height}_perf_vs_time.png")


def add_caption(fig, caption: str, fontsize: int = 10) -> None:
    """
    Ajoute une caption sous la figure (dans le canvas), pour que la figure soit self-contained.
    """
    fig.text(
        0.5, -0.08, caption,
        ha="center", va="top",
        fontsize=fontsize,
        wrap=True
    )

# -----------------------------
# 10) Lancement sur 3 tailles
# -----------------------------
if __name__ == "__main__":
    # Réglages conseillés (tu peux ajuster)
    COMMON = dict(
        n_trials=30,
        objective_seeds=[0, 1, 2],
        n_env_steps_train=20000,   # surtout pour 8x8
        eval_every=500,
        n_eval_episodes=30,
        n_runs=20,
        threshold=0.9,
        alpha_test=0.05,
        out_prefix="results",
    )

    run_benchmark_for_size(3, 3, **COMMON)
    # run_benchmark_for_size(5, 5, **COMMON)
    # run_benchmark_for_size(8, 8, **COMMON)

[I 2026-02-25 21:21:17,146] A new study created in memory with name: no-name-5af3185b-453b-4ccd-ba81-eb12891ad5a9
[I 2026-02-25 21:21:19,969] Trial 0 finished with value: 0.4374 and parameters: {'alpha': 0.35578899642343537, 'epsilon': 0.0974113806247247}. Best is trial 0 with value: 0.4374.
[I 2026-02-25 21:21:22,802] Trial 1 finished with value: 0.6561 and parameters: {'alpha': 0.04723689066575911, 'epsilon': 0.1276903115291674}. Best is trial 1 with value: 0.6561.
[I 2026-02-25 21:21:25,574] Trial 2 finished with value: 0.6561 and parameters: {'alpha': 0.0014763182272910908, 'epsilon': 0.14823545569018753}. Best is trial 1 with value: 0.6561.
[I 2026-02-25 21:21:28,051] Trial 3 finished with value: 0.4374 and parameters: {'alpha': 0.2047598568165943, 'epsilon': 0.10557726983434342}. Best is trial 1 with value: 0.6561.
[I 2026-02-25 21:21:30,405] Trial 4 finished with value: 0.6561 and parameters: {'alpha': 0.016783943470920798, 'epsilon': 0.21532503599744512}. Best is trial 1 with v